In [31]:
import numpy as np
import scanpy as sc
import pandas as pd

In [32]:
sc_data_path = "/data/qyyuan/project/ST_CP/JE/NC_review/Methods/data_stereo-seq/adata_sc_Xenium.h5ad"
st_data_path = "/data/qyyuan/project/ST_CP/JE/NC_review/Methods/data_stereo-seq/adata_st_Stereo-seq_drop.h5ad"

In [33]:
import scanpy as sc
import numpy as np
adata = sc.read_h5ad(sc_data_path)
common_genes = np.load('/home/qyyuan/project/ST_CP/JE/NC_review/Methods/Benchmark/sim_stereo-seq/Tangram_result/markers.npy', allow_pickle=True)
#adata=adata[:,common_genes]
sc.pp.filter_genes(adata, min_cells=1)
sc.pp.filter_cells(adata, min_genes=1)
adata.var['mt'] = adata.var_names.str.startswith('mt:rRNA')  

In [34]:
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
adata = adata[adata.obs.pct_counts_mt < 20, :]
print(f"filtered: {adata.shape[0]} cells × {adata.shape[1]} genes")
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,       
    batch_key=None,
    flavor='seurat_v3',        
    subset=True                
)
print(f"filtered: {adata.shape[0]} cells × {adata.shape[1]} genes")
sc.pp.scale(adata, max_value=10)  
# ----------------- 5.  PCA (100 ) -----------------
# run PCA
sc.tl.pca(
    adata, 
    svd_solver='arpack',     
    n_comps=100,              
    zero_center=True,        
    use_highly_variable=True, 
    random_state=42          
)
pca_scores = adata.obsm['X_pca']  # (n_cells, 100)
pca_loadings = adata.varm['PCs']   # (n_genes, 100)

filtered: 2489 cells × 7861 genes


/home/qyyuan/anaconda3/envs/GPU/lib/python3.11/site-packages/scanpy/preprocessing/_normalization.py:269: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)
/home/qyyuan/anaconda3/envs/GPU/lib/python3.11/site-packages/legacy_api_wrap/__init__.py:88: UserWarning: `flavor='seurat_v3'` expects raw count data, but non-integers were found.
  return fn(*args_all, **kw)


filtered: 2489 cells × 2000 genes


/home/qyyuan/anaconda3/envs/GPU/lib/python3.11/functools.py:909: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)
/home/qyyuan/anaconda3/envs/GPU/lib/python3.11/site-packages/scanpy/preprocessing/_pca/__init__.py:226: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(


In [35]:
pca_scores

array([[-0.5586429 , -4.260567  , -2.026977  , ...,  4.8067236 ,
        -0.26376742,  2.1220076 ],
       [-1.8274027 , -2.059138  , -0.7252353 , ..., -1.0799733 ,
         0.00906572,  0.86350197],
       [-1.0398232 , -0.72680926, -1.7052315 , ...,  0.50368035,
        -1.0462838 , -0.4774627 ],
       ...,
       [ 0.23049815, -1.7634178 , -1.0133544 , ..., -1.0958635 ,
        -5.3663387 ,  0.65536636],
       [-2.9507332 ,  6.801791  ,  4.0860596 , ...,  0.5145542 ,
        -0.3755441 , -1.0656149 ],
       [-2.2211368 ,  4.3259034 ,  1.8491768 , ..., -1.4340311 ,
        -2.6263819 ,  3.1483479 ]], shape=(2489, 100), dtype=float32)

In [36]:
import numpy as np
from sklearn.manifold import Isomap
from sklearn.neighbors import NearestNeighbors
from scipy.sparse.csgraph import shortest_path
from scipy.spatial.distance import pdist, squareform
import matplotlib.pyplot as plt
import time

def compute_geodesic_distances_isomap(X, n_neighbors=15, n_components=10, sample_fraction=0.05):
    """
    Compute approximate geodesic distance matrix for high-dimensional data using Isomap.
    
    Parameters:
        X: (n_samples, n_features) data matrix
        n_neighbors: number of k-nearest neighbors
        n_components: Isomap embedding dimension (for acceleration)
        sample_fraction: subsampling ratio (0~1), use 0.05~0.1 if memory is limited
    
    Returns:
        geodesic_dist: (n_samples_sub, n_samples_sub) geodesic distance matrix
        sample_idx: original indices of the subsampled points
    """
    if sample_fraction < 1.0:
        n_samples = X.shape[0]
        n_sub = int(n_samples * sample_fraction)
        np.random.seed(42)
        sample_idx = np.random.choice(n_samples, n_sub, replace=False)
        X_sub = X[sample_idx, :]
        print(f"Subsampled to {n_sub} samples")
    else:
        X_sub = X.copy()
        sample_idx = np.arange(X.shape[0])
        n_sub = X_sub.shape[0]
    
    # Use Isomap to approximate geodesic distances
    # Isomap builds a k-NN graph internally, then computes shortest paths
    print("Building Isomap model...")
    start_time = time.time()
    
    isomap = Isomap(
        n_neighbors=n_neighbors,
        n_components=n_components,
        metric='euclidean',
        n_jobs=-1
    )
    embedding = isomap.fit_transform(X_sub)
    
    # distance_matrix_ attribute contains geodesic distances
    if hasattr(isomap, 'distance_matrix_'):
        geodesic_dist = isomap.distance_matrix_
    else:
        # Fallback: manually compute shortest paths from k-NN graph
        from sklearn.neighbors import kneighbors_graph
        knn_graph = kneighbors_graph(X_sub, n_neighbors=n_neighbors, mode='distance', include_self=False)
        geodesic_dist = shortest_path(csgraph=knn_graph, directed=False, method='D')
    
    elapsed = time.time() - start_time
    print(f"Geodesic distance computation completed in {elapsed:.2f} seconds")
    print(f"Distance matrix shape: {geodesic_dist.shape}")
    
    return geodesic_dist, sample_idx




In [37]:
# ----------------- Usage Example -----------------
# X is your data matrix (19979, 1605)
# Subsampling is recommended due to large sample size
geodesic_dist, sample_idx = compute_geodesic_distances_isomap(
    pca_scores, 
    n_neighbors=20,
    n_components=30,
    sample_fraction=1
)

print(f"Geodesic distance - Mean: {geodesic_dist.mean():.4f}, Std: {geodesic_dist.std():.4f}")

Building Isomap model...
Geodesic distance computation completed in 4.14 seconds
Distance matrix shape: (2489, 2489)
Geodesic distance - Mean: 55.4833, Std: 17.2878


In [38]:
np.save('Ours/geodesic_dist_sub.npy', geodesic_dist)

In [39]:
np.save('Ours/sample_idx.npy', sample_idx)

In [40]:
np.save('Ours/pca_scores.npy',pca_scores)